In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)


In [4]:
from langchain_core.messages import HumanMessage
response=model.invoke([HumanMessage("what is the capital of india")])

In [5]:
response

AIMessage(content='The capital of India is New Delhi.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 41, 'total_tokens': 50, 'completion_time': 0.007138923, 'completion_tokens_details': None, 'prompt_time': 0.002166458, 'prompt_tokens_details': None, 'queue_time': 0.053046171, 'total_time': 0.009305381}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d80c3-3728-7ab0-9375-c7ecac817665-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 9, 'total_tokens': 50})

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()

    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [8]:
config={"configurable":{"session_id":"user1"}}
with_message_history.invoke([HumanMessage(content='what is a dog')],config=config)

AIMessage(content="A dog is a domesticated mammal that belongs to the family Canidae. Dogs are closely related to wolves and are known for their loyalty, intelligence, and ability to form strong bonds with humans.\n\nPhysical Characteristics:\n\n* Dogs come in a wide range of shapes and sizes, from small breeds like the Chihuahua to large breeds like the Great Dane.\n* They have a furry coat, a tail, and four legs.\n* Some breeds have erect ears, while others have floppy ears.\n* Dogs have a keen sense of smell and hearing.\n\nBehavior:\n\n* Dogs are social animals that thrive on interaction with their human family and other dogs.\n* They are known for their loyalty and affection towards their owners.\n* Dogs are highly trainable and can be trained to perform a variety of tasks, from simple obedience commands to complex tasks like search and rescue.\n* Some breeds are naturally more energetic and require more exercise than others.\n\nTypes of Dogs:\n\n* Sporting dogs (e.g. Golden Retri

prompts templates

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant answers all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="input")
    ]
)

chain=prompt|model

In [13]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input"
)

In [14]:
with_message_history.invoke(
    {"input": [HumanMessage(content="hi my name is abhi")]},
    config=config
)

AIMessage(content="Hello Abhi, it's nice to meet you. I'm happy to chat with you about anything you'd like. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 433, 'total_tokens': 466, 'completion_time': 0.126596319, 'completion_tokens_details': None, 'prompt_time': 0.050833385, 'prompt_tokens_details': None, 'queue_time': 0.121043666, 'total_time': 0.177429704}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d80d9-7620-79a2-aa68-c6329cbd2231-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 433, 'output_tokens': 33, 'total_tokens': 466})

In [15]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

In [18]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

chain=(
    RunnablePassthrough.assign(input=itemgetter("input")|trimmer)|prompt|model
)



In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input"
)
messages = [
    SystemMessage(content="You are helpful"),
    HumanMessage(content="Hi"),
    AIMessage(content="Hello!"),
    HumanMessage(content="Tell me a joke")
]




config={"configurable": {"session_id": "user1"}}
with_message_history.invoke({"input": messages}, config=config)
with_message_history.invoke({"input": "what was the last joke?"}, config=config)

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_core\language_models\base.py:336: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


AIMessage(content='The last joke I told you was: \n\nWhat do you call a fake noodle?\n\nAn impasta!\n\nDid you find it amusing?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 111, 'total_tokens': 140, 'completion_time': 0.090042547, 'completion_tokens_details': None, 'prompt_time': 0.007955786, 'prompt_tokens_details': None, 'queue_time': 0.051112344, 'total_time': 0.097998333}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d80e4-05e1-7eb0-94c6-d945170d1d65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 29, 'total_tokens': 140})

os.environ ek dictionary-like object hai jisme process ke saare environment variables stored hote hain. iska mtlb ye ki agr 4 process ho to sbka os.environ alg alg hoga kya?

Haan bhai, har process ka apna environment hota hai.

Maan le tere system pe 4 processes chal rahe hain:

Process A  (Python App)
Process B  (Chrome)
Process C  (VS Code)
Process D  (Redis)

Har process ke paas apni copy hoti hai environment variables ki.

Process A
  os.environ
    GROQ_API_KEY=abc

Process B
  os.environ
    (kuch aur values)

Process C
  os.environ
    PATH=...
    JAVA_HOME=...

Process D
  os.environ
    REDIS_PORT=6379

Ye memory share nahi karte.

Example

Python process me:

import os

os.environ["NAME"] = "Kajal"

Ab:

print(os.environ["NAME"])

to "Kajal" milega.

Lekin dusri terminal me naya Python run karo:

import os
print(os.getenv("NAME"))

to None aa sakta hai.

Kyuki pehle process ne apni local copy modify ki thi.

Process create hone par kya hota hai?

OS jab process start karta hai:

Operating System
        ↓
Create Process
        ↓
Copy Environment Variables
        ↓
Process os.environ

Matlab process ko startup pe environment mil jata hai.






Maan le tu chalata hai:

gunicorn app:app -w 4

Yahan:

Master Process
    │
    ├── Worker 1
    ├── Worker 2
    ├── Worker 3
    └── Worker 4

Total 5 processes hain (1 master + 4 workers).

os.environ ka kya hoga?

Jab Gunicorn start hota hai:

Shell
  │
  └── Gunicorn Master

Master process ko environment variables milte hain:

os.environ = {
    "GROQ_API_KEY": "abc123",
    "DB_URL": "postgres://..."
}

Phir master workers create karta hai.

Har worker ko environment variables ki copy mil jati hai.

Worker 1
    GROQ_API_KEY=abc123

Worker 2
    GROQ_API_KEY=abc123

Worker 3
    GROQ_API_KEY=abc123

Worker 4
    GROQ_API_KEY=abc123
Kya sabka os.environ same object hai?

❌ Nahi.

Har process ki alag memory hoti hai.

Example:

Worker 1 me:

os.environ["NAME"] = "worker1"

to sirf Worker 1 me change hoga.

Worker 2 me:

print(os.getenv("NAME"))

to None ya purani value milegi.

Kyuki:

Worker 1 Memory ≠ Worker 2 Memory
.env kab load hoti hai?

Maan le:

load_dotenv()

app = FastAPI()

Gunicorn jab worker start karega to har worker process ye code execute karega.

Worker 1 → load_dotenv()
Worker 2 → load_dotenv()
Worker 3 → load_dotenv()
Worker 4 → load_dotenv()

Aur har worker apne os.environ me values load kar lega.

Important Interview Point

Agar tu worker me koi global variable rakhe:

counter = 0

aur request aaye:

counter += 1

to:

Worker 1 counter = 10
Worker 2 counter = 5
Worker 3 counter = 20
Worker 4 counter = 7

Kyuki memory share nahi hoti.

Isi liye Gunicorn multi-worker setup me shared state ke liye:

Redis
Database
Memcached

use karte hain.

Short Answer

Gunicorn ke 4 workers me:

Worker 1 -> os.environ (copy)
Worker 2 -> os.environ (copy)
Worker 3 -> os.environ (copy)
Worker 4 -> os.environ (copy)

Environment variables same values se start hote hain, lekin har worker ka os.environ alag memory object hota hai. Ek worker me change karoge to dusre workers me reflect nahi hoga. 🚀

Step 1
store = {}

Ye ek simple Python dictionary hai.

Initially:

store = {}

Khaali.

Isme hum session-wise chat history rakhenge.

Baad me aisa ho jayega:

store = {
    "user1": ChatMessageHistory(...),
    "user2": ChatMessageHistory(...)
}
Step 2
def get_session_history(session_id: str) -> BaseChatMessageHistory:

Ye function session_id leta hai.

Example:

get_session_history("user1")
Step 3
if session_id not in store:
    store[session_id] = ChatMessageHistory()

Agar pehli baar user aaya:

store = {}

aur:

get_session_history("user1")

to:

store["user1"] = ChatMessageHistory()

ban jayega.

Result:

store = {
    "user1": ChatMessageHistory()
}
Step 4
return store[session_id]

User ki history return kar do.

Example:

ChatMessageHistory(
    messages=[]
)
ChatMessageHistory kya hai?

Ye ek object hai jo messages store karta hai.

Example:

history = ChatMessageHistory()

history.add_user_message("Hi")
history.add_ai_message("Hello")

Ab:

history.messages

Output:

[
    HumanMessage("Hi"),
    AIMessage("Hello")
]
Step 5
with_message_history = RunnableWithMessageHistory(
    model,
    get_session_history
)

Yahan:

model

normal LLM hai.

Example:

model = ChatGroq(...)

Ab LangChain bol raha hai:

"LLM ko memory ke saath wrap kar do."

Without memory:

model.invoke("Hi")

LLM ko sirf current message dikhega.

With memory:

with_message_history.invoke(...)

LLM ko purani conversation bhi dikhegi.

Step 6
config = {
    "configurable": {
        "session_id": "user1"
    }
}

Ye batata hai:

Kis user ki memory use karni hai?

Current user:

"user1"
Step 7
with_message_history.invoke(
    [HumanMessage(content="what is a dog")],
    config=config
)

Flow:

A

Session id nikala:

"user1"
B

Call hua:

get_session_history("user1")

Store me nahi mila.

To create hua:

store = {
    "user1": ChatMessageHistory()
}
C

User message add hua:

HumanMessage(
    content="what is a dog"
)

History:

[
    HumanMessage("what is a dog")
]
D

LLM ko bheja gaya:

Human: what is a dog

LLM answer:

A dog is a domesticated mammal...
E

AI response bhi history me save ho gaya.

Ab history:

[
    HumanMessage("what is a dog"),
    AIMessage("A dog is a domesticated mammal...")
]
Ab second request
with_message_history.invoke(
    [HumanMessage(content="and what does it eat?")],
    config=config
)

Session id same:

"user1"

To existing history mil gayi.

History pehle se:

[
    HumanMessage("what is a dog"),
    AIMessage("A dog is a domesticated mammal...")
]

Ab new message add hua:

HumanMessage("and what does it eat?")

LLM ko ye pura context dikhega:

Human: what is a dog

AI: A dog is a domesticated mammal...

Human: and what does it eat?

Isliye model samajh jayega ki "it" = dog.

Store ka final state
store = {
    "user1": ChatMessageHistory(
        messages=[
            HumanMessage("what is a dog"),
            AIMessage("A dog is a domesticated mammal..."),
            HumanMessage("and what does it eat?"),
            AIMessage("Dogs typically eat...")
        ]
    )
}
Production me problem

Ye memory sirf RAM me hai:

store = {}

Agar:

Ctrl + C

kar diya ya Gunicorn restart ho gaya:

store = {}

sab history gayab.

Isliye production me:

RedisChatMessageHistory
Postgres
MongoDB

use karte hain.

Short Summary
store

→ session-wise chat histories store karta hai.

get_session_history(session_id)

→ user ki history return karta hai.

RunnableWithMessageHistory

→ LLM ke saath memory attach karta hai.

session_id="user1"

→ batata hai kis user ki memory use karni hai.

invoke(...)

→ user message history me add karta hai, LLM ko purani conversation ke saath bhejta hai, aur response bhi history me save kar deta hai. 🚀

MessagesPlaceholder(variable_name="input")

Purpose:
- Prompt me dynamic list of chat messages insert karne ke liye use hota hai.
- Jab input ek string nahi balki messages (HumanMessage, AIMessage, SystemMessage) ki list ho tab use karte hain.
- RunnableWithMessageHistory ke saath commonly use hota hai, kyunki history + current message automatically inject ho jate hain.

Example:

Prompt:
[
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("input")
]

Input:
{
    "input": [
        HumanMessage("Hi"),
        AIMessage("Hello"),
        HumanMessage("My name is Abhi")
    ]
}

Final Prompt:
System: You are a helpful assistant
Human: Hi
AI: Hello
Human: My name is Abhi

Difference:
- ("human", "{input}") → input should be a string.
- MessagesPlaceholder("input") → input should be a list of messages.

Use Case:
- Chat history
- Conversation memory
- RunnableWithMessageHistory
- Multi-turn conversations

One-line interview answer:

MessagesPlaceholder is used to dynamically inject a list of chat messages (conversation history) into a prompt, especially when working with memory and chat-based applications.

Example:

from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("messages")
])

chain = prompt | model

Ab messages variable expect ho raha hai.

RunnableWithMessageHistory:

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

Invoke:

with_message_history.invoke(
    {
        "messages": [HumanMessage(content="Hi")]
    },
    config=config
)

Lekin agar tum bilkul direct message/list dena chahte ho:

with_message_history.invoke(
    [HumanMessage(content="Hi")],
    config=config
)

to usually ChatPromptTemplate + MessagesPlaceholder wala pattern nahi use karte.

Seedha model ko history wrapper ke saath wrap kar dete hain:

chain = model

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history
)

Aur invoke:

with_message_history.invoke(
    [HumanMessage(content="Hi")],
    config=config
)

Yahan history wrapper existing history + new message ko merge karke model ko de dega.

Simple rule
Prompt me MessagesPlaceholder use kar rahe ho?
→ Dictionary + input_messages_key use karo.

Sirf chat model use kar rahe ho?
→ Direct messages/list pass kar sakte ho.